In [2]:
import sys
import os
from pathlib import Path
project_dir = Path(os.path.abspath('')).parent
sys.path.insert(0, project_dir.as_posix())

from tqdm import trange, tqdm
import numpy as np
import torch
from matplotlib import pyplot as plt

%load_ext autoreload
%autoreload 2
import train_dpr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Train

In [ ]:
train_args = train_dpr.TrainArgs(
    save_weight="dpr",  # 保存时的前缀
    epochs=1,
    batch_size=32,
    accumulation_steps=1,
    learning_rate=4e-6,
    hidden_size=512,
    num_hidden_layers=8,
    max_query_len=64,  # 问题最大长度
    max_passage_len=128,  # 单个document的最大长度，对于`natural-questions`数据集，建议设置为512
    from_weight="pretrain",  # 从MiniMind pretrain 512的权重开始训练
    train_bert=False,  # 是否训练BERT-Mini，若是则加载BERT，则MiniMind模型和权重均失效
    use_wandb=True,
    use_swanlab=True,  # 使用swanlab时需要同时设置use_wandb=True, wandb会被替换为swanlab
    dtype="float16",
    dataset="ChineseSquad",  # 训练数据集（同时也是检索数据集），`ChineseSquad`或`natural-questions`
    wandb_project="MiniMind-DPR",
)
# 加载数据
model, tokenizer, train_ds, test_ds = train_dpr.train(train_args, early_return=1)

所加载Model可训练参数：25.830 百万


In [ ]:
train_dpr.train(train_args, model, tokenizer, train_ds)

### Evaluate

In [ ]:
eval_args = train_dpr.EvalArgs(
    weight="full_sft",  # 加载MiniMind full_sft_768权重
    hidden_size=768,  # chat模型隐藏层维度(与retriever无关)
    num_hidden_layers=16,  # chat模型隐藏层数量(与retriever无关)
    top_k=20,  # 检索时返回的文档数量，reranker会从中选择最终答案
    rag=True,  # 是否开启RAG
    historys=0,
    use_sbert_retriever=True,  # 使用sentence-transformer作为retriever，检索成功率会大幅提升
)

# ----- 下面的代码只有在rag=True时才需要，否则可令retriever=reranker=docs=doc_embeddings=None -----

# 若想使用自行训练的Minimind retriever，可将retriever和tokenizer传入init_rag函数
# 例如: retriever, tokenizer, _, _ = train_dpr.train(train_args, early_return=1)
# 这里设为None是因为用了SBERT retriever
# retriever = tokenizer = None
dataset = train_dpr.load_dataset(
    "ChineseSquad",
    max_query_len=eval_args.max_query_len,
    max_passage_len=eval_args.max_passage_len,
)
docs = []
for key in ["train", "validation"]:
    for doc in dataset[key]["context"]:
        if doc not in docs:  # 文档去重
            docs.append(doc)
retriever, reranker, doc_embeddings = train_dpr.init_rag(
    device, docs, model, tokenizer, eval_args
)

In [ ]:
print(doc_embeddings.shape)
print(docs[0])

torch.Size([2248, 384])
2012年1月7日，碧昂丝在纽约莱诺克斯山医院生下了她的第一个孩子，一个女儿，蓝色的艾薇·卡特。5个月后，她在Revel Atlantic City的Ovation Hall演出了4个晚上，庆祝度假村开业，这是她生下蓝色常春藤后的首次演出。


In [19]:
eval_args.rag = True
train_dpr.evaluate(eval_args, retriever, reranker, docs, doc_embeddings)

所加载Model可训练参数：104.031 百万
👶: 估计有4.88亿至5.35亿人信奉什么宗教
🤖️:根据估计，4.88亿至5.35亿人信奉佛教。

👶: 南安普敦机场在哪个城镇
🤖️:南安普顿机场位于伊斯特利镇，它是一个地区性机场。

👶: 路德什么时候死的
🤖️:路德在1546年凌晨2点45分，他死于中风，享年62岁。

👶: 超级碗开幕之夜于何时在何地举行
🤖️:超级碗开幕之夜于2016年2月1日在圣何塞SAP中心举行。

